# ATiG 2026: LD score regression
29 October 2026 · About 75 minutes · Work in pairs

We estimate SNP heritability and genetic correlations from published GWAS results with
LD score regression (LDSC). Run the cells in order, discuss the questions and change the
settings marked `# TRY`. [Open in Colab](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-10-29/LDSC/LDSC_practical.ipynb) and choose
File → Save a copy in Drive.

## Setup
Run the three cells. The first installs [PLDSC](https://github.com/bvilhjal/PLDSC) and
[allelign](https://github.com/bvilhjal/allelign); the last downloads the GWAS files (about 240 MB).

In [ ]:
%pip install -q "pldsc @ git+https://github.com/bvilhjal/PLDSC@v0.1.1"

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import allelign
from pldsc import load_ld_reference, ldsc_h2_sumstats, ldsc_rg_sumstats

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
#@title Download the GWAS files (no editing needed)
URLS = {
    'BMI': 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST006001-GCST007000/'
           'GCST006900/Meta-analysis_Locke_et_al+UKBiobank_2018_UPDATED.txt.gz',
    'TG': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TG.txt.gz',
    'DBP': 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST006001-GCST007000/'
           'GCST006630/Evangelou_30224653_DBP.txt.gz',
}
folder = Path('gwas_downloads')
folder.mkdir(exist_ok=True)

paths = {}
for trait, url in URLS.items():
    paths[trait] = folder / f'{trait}.txt.gz'
    if not paths[trait].exists():
        urllib.request.urlretrieve(url, paths[trait])
    print(trait, f'{paths[trait].stat().st_size / 1e6:.0f} MB')

## Q1. Look at the files

BMI from [Yengo et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30124842/) (about 700,000 people),
triglycerides (TG) from [Willer et al. 2013](https://pubmed.ncbi.nlm.nih.gov/24097068/) (about 90,000)
and diastolic blood pressure (DBP) from [Evangelou et al. 2018](https://pubmed.ncbi.nlm.nih.gov/30224653/)
(about 750,000).

Find the effect allele, other allele, effect, standard error, sample size and position in
each file. How do the files differ?

In [ ]:
for trait, path in paths.items():
    print(f'The first rows of the {trait} file:')
    display(pd.read_csv(path, sep=r'\s+', nrows=3, dtype=str))

## Q2. Read the files

allelign recognises common column names (Table 1). The DBP file has no rsIDs, so we take
the positions from `MarkerName`. For DBP, allelign uses `N_effective`, the sample size
times imputation quality.

Which columns were not found? Why does N matter for heritability?

In [ ]:
columns = {}
for trait, path in paths.items():
    columns[trait] = allelign.inspect_columns(path)['mapping']

print('Table 1. File columns used by allelign; blank if not found.')
display(pd.DataFrame(columns).fillna(''))

In [ ]:
gwas = {}
for trait, path in paths.items():  # About 30 seconds.
    gwas[trait] = allelign.read_sumstats(path)
    print(f'{trait}: {len(gwas[trait]):,} variants, median N {np.median(gwas[trait].n_eff):,.0f}')

# DBP has no rsIDs: MarkerName is chromosome:position:type, e.g. 10:100000625:SNP.
parts = pd.Series(gwas['DBP'].id).str.split(':', expand=True)
gwas['DBP'].chrom = parts[0].to_numpy()
gwas['DBP'].pos = parts[1].astype(int).to_numpy()

## Q3. Match to the LD reference

PLDSC comes with LD scores for about 1.2 million HapMap3 SNPs, calculated from 489
Europeans in the 1000 Genomes Project. It leaves out about 30,000 of them in regions with
long-range LD, such as the MHC. Each GWAS is aligned to the reference allele `a1`
(Table 2), giving one table per trait (Table 3).

Why is so little of the DBP file used? Table 4 follows rs1421085 in *FTO*. Why must the
traits refer to the same allele? Figure 1 shows the strongest 1% of SNPs. Which trait has
the most associated regions?

In [ ]:
ref = load_ld_reference('hm3-eur')
print(ref)

aligned = {}
counts = {}
for trait, sumstats in gwas.items():
    keep, qc = allelign.qc_sumstats(sumstats)
    aligned[trait] = allelign.harmonize(sumstats.subset(keep), ref.variants)
    counts[trait] = {'in file': len(sumstats), 'after QC': qc['n_kept'],
                     'matched': aligned[trait].log['n_matched'],
                     'sign changed': aligned[trait].log['n_flipped']}

print('Table 2. Variants kept at each step.')
display(pd.DataFrame(counts))

In [ ]:
def aligned_table(trait):
    """One row per aligned SNP, with its z score for allele a1, N and LD score."""
    h = aligned[trait]
    rows = h.var_index  # Rows of the LD reference.
    return pd.DataFrame({'snp': ref.variants.id[rows],
                         'chrom': ref.variants.chrom[rows].astype(int),
                         'pos': ref.variants.pos[rows],
                         'a1': ref.variants.a1[rows],
                         'a2': ref.variants.a2[rows],
                         'z': h.beta / h.se,
                         'n': h.n_eff,
                         'ld_score': ref.scores[rows]})


snps = {}
for trait in aligned:
    snps[trait] = aligned_table(trait)

print('Table 3. The first aligned SNPs for BMI.')
display(snps['BMI'].head())

In [ ]:
snp = 'rs1421085'  # TRY: rs1260326 (GCKR) or rs13107325 (SLC39A8)

rows = {}
for trait, sumstats in gwas.items():
    after = snps[trait].set_index('snp').loc[snp]
    # Find the SNP in the file: by rsID, or by position (DBP).
    in_file = (sumstats.id == snp) | ((sumstats.chrom == str(after['chrom'])) & (sumstats.pos == after['pos']))
    i = np.flatnonzero(in_file)[0]
    rows[trait] = {'effect allele in file': sumstats.ea[i],
                   'z in file': round(sumstats.beta[i] / sumstats.se[i], 2),
                   'a1': after['a1'],
                   'z for a1': round(after['z'], 2)}

print(f'Table 4. {snp} before and after alignment.')
display(pd.DataFrame(rows).T)

In [ ]:
def manhattan(ax, table, top=0.01, cap=50):
    """Manhattan plot of the strongest 1% of SNPs; -log10 P above `cap` is drawn at `cap`."""
    minus_log10_p = -stats.chi2.logsf(table['z']**2, df=1) / np.log(10)  # Works for huge z.
    best = table[minus_log10_p >= np.quantile(minus_log10_p, 1 - top)]
    x = best['chrom'] + best['pos'] / 250e6  # Chromosomes side by side; chr1 is 249 Mb.
    colour = np.where(best['chrom'] % 2 == 0, 'grey', 'steelblue')
    y = np.minimum(minus_log10_p[best.index], cap)
    ax.scatter(x, y, c=colour, s=4)
    ax.axhline(-np.log10(5e-8), color='red', linewidth=1)  # Genome-wide significance.
    ax.set_xticks(np.arange(1, 23) + 0.4, np.arange(1, 23))
    ax.set_ylabel('−log10 P')


fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, trait in zip(axes, snps):
    manhattan(ax, snps[trait])
    ax.set_title(trait)
axes[-1].set_xlabel('Chromosome')
fig.suptitle('Figure 1. The strongest 1% of SNPs (capped at 50); red line: P = 5e-8')
plt.tight_layout()
plt.show()

## Q4. Estimate SNP heritability

LDSC models the χ² statistic of SNP $j$ as

$$E[\chi^2_j] = 1 + Na + \frac{N h^2}{M}\,\ell_j, \qquad (1)$$

with LD score $\ell_j$, sample size $N$, number of SNPs $M$ and confounding $a$.

Which trait has the highest heritability (Table 5, Figure 2)? Compare the intercepts with
λGC, the median χ² divided by 0.455. Why is TG's intercept below one? Set
`M = ref.m_snps` and rerun both cells: what changes?

In [ ]:
M = ref.m_snps_5_50  # SNPs with MAF 5–50%, as in the original LDSC. TRY: ref.m_snps

fits = {}
h2_table = {}
for trait, sumstats in gwas.items():  # About 30 seconds.
    fit = ldsc_h2_sumstats(sumstats, ref.variants, ref.scores, m_snps=M).estimate
    fits[trait] = fit
    h2_table[trait] = {'h2': fit.h2, 'SE': fit.h2_se,
                       'intercept': fit.intercept, 'intercept SE': fit.intercept_se,
                       'mean chi2': fit.mean_chisq,
                       'lambda GC': allelign.genomic_inflation(z=snps[trait]['z'].to_numpy())}

print(f'Table 5. SNP heritability with M = {M:,}.')
display(pd.DataFrame(h2_table).T)

In [ ]:
def used_in_fit(z, n):
    """SNPs that LDSC keeps: chi2 at most max(80, N/1000)."""
    return z**2 <= np.maximum(80, n / 1000)


def binned_means(x, y, bins=50):
    """Means of x and y in bins with equal numbers of SNPs, sorted by x."""
    data = pd.DataFrame({'x': x, 'y': y})
    data = data[data['x'] < data['x'].quantile(0.99)]  # Leave out the top 1% of x.
    data['bin'] = pd.qcut(data['x'], bins, labels=False)
    return data.groupby('bin').mean()


fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, trait in zip(axes, snps):
    table = snps[trait][used_in_fit(snps[trait]['z'], snps[trait]['n'])]
    means = binned_means(table['ld_score'], table['z']**2)
    ax.scatter(means['x'], means['y'], s=10)

    # Equation (1) with the fitted intercept and h2, at the median N.
    fit = fits[trait]
    ld_score = np.linspace(0, means['x'].max(), 100)
    ax.plot(ld_score, fit.intercept + table['n'].median() * fit.h2 / M * ld_score, color='darkorange')
    ax.set_title(trait)
    ax.set_xlabel('LD score')
axes[0].set_ylabel('Mean χ²')
fig.suptitle('Figure 2. Mean χ² in LD-score bins, with the LDSC fit (orange)')
plt.tight_layout()
plt.show()

## Q5. BMI and TG

Bivariate LDSC regresses the product of the two z scores on LD score:

$$E[z_{1j} z_{2j}] = \frac{\sqrt{N_1 N_2}\,\rho_g}{M}\,\ell_j + \frac{N_s\,\rho}{\sqrt{N_1 N_2}}, \qquad (2)$$

where $\rho_g$ is the genetic covariance, $N_s$ the number of people in both GWAS and $\rho$
their phenotypic correlation. The genetic correlation is $r_g = \rho_g / \sqrt{h^2_1 h^2_2}$.

Is $r_g$ clearly above zero? What does the intercept tell you? Why is the plain correlation
of z scores so much smaller? Where does the line in Figure 3 meet the y-axis?

In [ ]:
rg_bmi_tg = ldsc_rg_sumstats(gwas['BMI'], gwas['TG'], ref.variants, ref.scores,
                             m_snps=M, ncores=2).estimate
print(f'BMI–TG genetic correlation: {rg_bmi_tg.rg:.3f} (SE {rg_bmi_tg.rg_se:.3f})')
print(f'Genetic covariance: {rg_bmi_tg.gcov:.4f}')
print(f'Cross-trait intercept: {rg_bmi_tg.gcov_intercept:.3f} (SE {rg_bmi_tg.gcov_intercept_se:.3f})')

# For comparison: the plain correlation of z scores, ignoring LD.
both = snps['BMI'].merge(snps['TG'], on='snp', suffixes=('_bmi', '_tg'))
print(f"Correlation of z scores over {len(both):,} SNPs: {np.corrcoef(both['z_bmi'], both['z_tg'])[0, 1]:.3f}")

In [ ]:
used = used_in_fit(both['z_bmi'], both['n_bmi']) & used_in_fit(both['z_tg'], both['n_tg'])
table = both[used]
means = binned_means(table['ld_score_bmi'], table['z_bmi'] * table['z_tg'])

# Equation (2) with the fitted genetic covariance and intercept, at the median Ns.
slope = np.sqrt(table['n_bmi'].median() * table['n_tg'].median()) * rg_bmi_tg.gcov / M
ld_score = np.linspace(0, means['x'].max(), 100)

plt.figure(figsize=(5, 3.5))
plt.scatter(means['x'], means['y'], s=10)
plt.plot(ld_score, rg_bmi_tg.gcov_intercept + slope * ld_score, color='darkorange')
plt.axhline(0, color='grey', linestyle=':')
plt.xlabel('LD score')
plt.ylabel('Mean z(BMI) × z(TG)')
plt.title('Figure 3. BMI–TG z-score products, with the fit (orange)')
plt.tight_layout()
plt.show()

## Q6. Add blood pressure

UK Biobank is part of both the BMI and the DBP GWAS, and DBP was adjusted for BMI
(Table 6, Figure 4).

Did you expect the BMI–DBP result? How can adjusting for BMI change it? Why is its
cross-trait intercept close to zero?

In [ ]:
rg_table = {}
for trait1, trait2 in [('BMI', 'TG'), ('BMI', 'DBP'), ('TG', 'DBP')]:  # About a minute.
    fit = ldsc_rg_sumstats(gwas[trait1], gwas[trait2], ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    rg_table[f'{trait1}–{trait2}'] = {'rg': fit.rg, 'SE': fit.rg_se,
                                      'cross-trait intercept': fit.gcov_intercept,
                                      'intercept SE': fit.gcov_intercept_se}

results = pd.DataFrame(rg_table).T
print('Table 6. Genetic correlations.')
display(results)

plt.figure(figsize=(5, 2.5))
plt.errorbar(results['rg'], results.index, xerr=1.96 * results['SE'], fmt='o')
plt.gca().invert_yaxis()  # Same order as Table 6.
plt.axvline(0, color='grey', linestyle='--')
plt.xlabel('Genetic correlation, with 95% interval')
plt.title('Figure 4. Genetic correlations')
plt.tight_layout()
plt.show()

## Q7. Conclude

Write three sentences: the BMI–TG $r_g$ with its standard error, why it does not show that
BMI raises TG, and what the BMI–DBP result teaches you about published GWAS.

## Going further (optional)

- Run the cell below to add HDL, LDL, total cholesterol or height. Other European GWAS with
  rsIDs or GRCh37 positions work too, for example from the
  [GWAS Catalog](https://www.ebi.ac.uk/gwas/downloads/summary-statistics).
- Ask an AI model to write LDSC from scratch in NumPy, using the columns `z`, `n` and
  `ld_score` of `snps['BMI']`, and compare with Table 5. Does it treat M, the weights and
  very large χ² the way PLDSC does?

In [ ]:
MORE = {  # Same formats as TG and BMI.
    'HDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_HDL.txt.gz',
    'LDL': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_LDL.txt.gz',
    'TC': 'https://csg.sph.umich.edu/willer/public/lipids2013/jointGwasMc_TC.txt.gz',
    'Height': 'https://ftp.ebi.ac.uk/pub/databases/gwas/summary_statistics/GCST006001-GCST007000/'
              'GCST006901/Meta-analysis_Wood_et_al+UKBiobank_2018.txt.gz',
}
trait = 'HDL'  # TRY: 'LDL', 'TC' or 'Height'

path = folder / f'{trait}.txt.gz'
if not path.exists():
    urllib.request.urlretrieve(MORE[trait], path)
gwas[trait] = allelign.read_sumstats(path)

print(trait, ldsc_h2_sumstats(gwas[trait], ref.variants, ref.scores, m_snps=M).estimate)
for other in ['BMI', 'TG', 'DBP']:
    fit = ldsc_rg_sumstats(gwas[other], gwas[trait], ref.variants, ref.scores,
                           m_snps=M, ncores=2).estimate
    print(f'{other}–{trait}: rg {fit.rg:.3f} (SE {fit.rg_se:.3f})')

## Data and methods

BMI ([GCST006900](https://www.ebi.ac.uk/gwas/studies/GCST006900)), DBP
([GCST006630](https://www.ebi.ac.uk/gwas/studies/GCST006630)) and TG
([GLGC](https://csg.sph.umich.edu/willer/public/lipids2013/)) are European-ancestry
meta-analyses on GRCh37; see [`data/sources.json`](data/sources.json). The TG and DBP results
were corrected for inflation before release. The LD scores are the 1000 Genomes Phase 3
European scores released with LDSC ([Zenodo 7768714](https://doi.org/10.5281/zenodo.7768714)).
PLDSC fits equations (1) and (2) in one step and drops SNPs with χ² above max(80, N/1000).
The original LDSC software uses a two-step estimator, so its estimates differ a little,
most for TG. Background: [Bulik-Sullivan et al. 2015a](https://pubmed.ncbi.nlm.nih.gov/25642630/)
and [2015b](https://pubmed.ncbi.nlm.nih.gov/26414676/).